# HiSA Pre-Attention Implementation

This notebook builds the pre-attention HiSA stack in a reproducible, step-by-step order: config, math primitives, path encoding, direct-sum subspace rotation, low-rank projection, SGAM, parser integration, and final model assembly.

**Goal:** implement the structural positioning layer before attention so a token is encoded with `(m, d, p, s)` rather than only sequence position.

In [ ]:
# Section 0: pip installs and imports
%pip install -q torch numpy matplotlib seaborn pandas
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path.cwd().resolve()
for candidate in [repo_root, repo_root.parent, repo_root / 'hise', repo_root.parent / 'hise']:
    source_dir = candidate / 'src' if candidate.name != 'src' else candidate
    if source_dir.exists():
        sys.path.insert(0, str(source_dir))
    if (candidate / 'hise' / 'src').exists():
        sys.path.insert(0, str(candidate / 'hise' / 'src'))

print('repo_root =', repo_root)
print('torch', torch.__version__)

In [ ]:
# Section 1: HiSAConfig
from hise.config import HiSAConfig

config = HiSAConfig(d_model=768, alpha=0.5, beta=0.2, gamma=0.2, delta=0.1, rank=64)
print(config)
print('subspace sizes =', config.subspace_sizes)
print('d_m, d_d, d_p, d_s =', config.d_m, config.d_d, config.d_p, config.d_s)

In [ ]:
# Section 2: Math primitives
import torch
from hise.encoding.utils import complex_multiply, complex_power, normalize_to_unit_circle, rope_freqs, apply_rope

a = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b = torch.tensor([[0.0, 1.0], [1.0, 0.0]])
print('complex_multiply ->', complex_multiply(a, b))
print('complex_power(0) ->', complex_power(torch.tensor([[1.0, 0.0], [0.0, 1.0]]), 0))
print('rope_freqs(8) ->', rope_freqs(8, 10000.0))

positions = torch.tensor([[0, 1, 2], [3, 4, 5]])
x = torch.randn(2, 3, 8)
print('apply_rope shape ->', apply_rope(x, rope_freqs(8, 10000.0), positions).shape)

In [ ]:
# Section 3: PathEncoder
import torch
from hise.encoding.path_encoder import PathEncoder

encoder = PathEncoder(d_struct=154)
path_a = [0, 1, 0]
path_b = [0, 0, 1]
rot_a = encoder.encode_path(path_a)
rot_b = encoder.encode_path(path_b)
print('path_a =', path_a)
print('path_b =', path_b)
print('collision check =', bool(torch.allclose(rot_a, rot_b)))
print('rot_a[:6] =', rot_a[:6])

In [ ]:
# Section 4: DirectSumHiSA
from hise.encoding.subspace import DirectSumSubspace

model = DirectSumSubspace(config)
inputs = torch.randn(2, 16, 768)
structural_address = {
    'm': torch.arange(16).unsqueeze(0).expand(2, -1),
    'd': torch.zeros(2, 16, dtype=torch.long),
    'p': [[[] for _ in range(16)] for _ in range(2)],
    's': torch.zeros(2, 16, dtype=torch.long),
}
out = model(inputs, structural_address)
print('output shape =', out.shape)
print('output norm =', out.norm(dim=-1).mean().item())

In [ ]:
# Section 5: Critical tests
import torch
from hise.config import HiSAConfig
from hise.encoding.subspace import DirectSumSubspace
from hise.encoding.utils import apply_rope, rope_freqs
from hise.encoding.path_encoder import PathEncoder

# Degenerate: zero structural coordinates must reduce to RoPE
config = HiSAConfig(d_model=768)
model = DirectSumSubspace(config)
x = torch.randn(2, 16, 768)
positions = torch.arange(16).unsqueeze(0).expand(2, -1)
addr_zero = {'m': positions, 'd': torch.zeros(2, 16, dtype=torch.long), 'p': [[[] for _ in range(16)] for _ in range(2)], 's': torch.zeros(2, 16, dtype=torch.long)}
hisa_out = model(x, addr_zero)
rope_out = apply_rope(x[..., :config.d_m], rope_freqs(config.d_m, config.rope_base, d_model=config.d_model), positions)
print('degenerate test =', bool(torch.allclose(hisa_out[..., :config.d_m], rope_out, atol=1e-6)))

# No-collision path test
encoder = PathEncoder(d_struct=154)
rot_a = encoder.encode_path([0, 1, 0])
rot_b = encoder.encode_path([0, 0, 1])
print('collision test =', bool(not torch.allclose(rot_a, rot_b, atol=1e-6)))

# Invertibility test
parent_path = [0, 1]
child_sibling_idx = 2
child_path = parent_path + [child_sibling_idx]
parent_rot = encoder.encode_path(parent_path)
child_rot = encoder.encode_path(child_path)
recovered = encoder.get_parent_rotation(child_rot, child_sibling_idx)
print('invertibility test =', bool(torch.allclose(parent_rot, recovered, atol=1e-5)))

In [ ]:
# Section 6: LowRankStructuralProjection
import torch
from hise.model.low_rank import LowRankStructuralProjection

proj = LowRankStructuralProjection(d_struct=768, rank=64)
struct = torch.randn(4, 768)
rotation_fn = lambda z: z * 0.5
out = proj(struct, rotation_fn)
print('low-rank output shape =', out.shape)
print('rank compression factor =', 768 / 64)
print('effective FLOP reduction ~= ', (768 * 768) / (768 * 64 + 64 * 768))

In [ ]:
# Section 7: SGAMLayer
import torch
from hise.address import StructuralAddress

class SGAMLayer(torch.nn.Module):
    def __init__(self, depth_weight=1.0, parent_weight=1.0, sibling_weight=1.0):
        super().__init__()
        self.depth_weight = depth_weight
        self.parent_weight = parent_weight
        self.sibling_weight = sibling_weight

    def forward(self, addresses):
        tensor = torch.tensor([a.as_vector() for a in addresses], dtype=torch.float32)
        diffs = tensor[:, None, :] - tensor[None, :, :]
        depth = diffs[..., 1].abs() * self.depth_weight
        parent = (diffs[..., 2] != 0).float() * self.parent_weight
        sibling = diffs[..., 3].abs() * self.sibling_weight
        return depth + parent + sibling

addresses = [
    StructuralAddress(0, 1, 10, 0),
    StructuralAddress(1, 1, 10, 1),
    StructuralAddress(2, 2, 20, 0),
]
mask = SGAMLayer()(addresses)
print(mask)

In [ ]:
# Section 8: Markdown parser demo
from hise.parsers import parse_markdown

markdown = """# Report
## Risks
Revenue increased 47% year over year.
## Performance
Operating margin improved 8%.
"""
items = parse_markdown(markdown)
print(items)
for content, addr in items:
    print(content, '->', addr.as_tuple())

In [ ]:
# Section 9: End-to-end forward pass on parsed document
import torch
from hise.parsers import parse_markdown
from hise.encoding.subspace import DirectSumSubspace
from hise.config import HiSAConfig

markdown = '# Report
## Risks
Revenue increased 47% year over year.
## Performance
Operating margin improved 8%.'
items = parse_markdown(markdown)
texts = [item[0] for item in items]
addresses = [item[1] for item in items]
model = DirectSumSubspace(HiSAConfig(d_model=768))
base = torch.randn(len(items), 768)
struct = {
    'm': torch.tensor([addr.sequence_position for addr in addresses], dtype=torch.long).unsqueeze(0).expand(1, -1),
    'd': torch.tensor([addr.depth for addr in addresses], dtype=torch.long).unsqueeze(0),
    'p': [[[] for _ in addresses]],
    's': torch.tensor([addr.sibling_index for addr in addresses], dtype=torch.long).unsqueeze(0),
}
out = model(base.unsqueeze(0), struct)
print('tokens =', len(items))
print('forward output shape =', out.shape)

In [ ]:
# Section 10: Visualization — subspace rotation before vs after
import matplotlib.pyplot as plt
import torch
from hise.config import HiSAConfig
from hise.encoding.subspace import DirectSumSubspace

config = HiSAConfig(d_model=768)
model = DirectSumSubspace(config)
base = torch.randn(1, 16, 768)
addr = {'m': torch.arange(16).unsqueeze(0), 'd': torch.zeros(1, 16, dtype=torch.long), 'p': [[[] for _ in range(16)]], 's': torch.zeros(1, 16, dtype=torch.long)}
out = model(base, addr)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(base[0, 0].detach().numpy(), label='before')
axes[1].plot(out[0, 0].detach().numpy(), label='after')
axes[0].set_title('Before HiSA rotation')
axes[1].set_title('After HiSA rotation')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Section 11: SGAM bias matrix visualization
import matplotlib.pyplot as plt
import torch
from hise.address import StructuralAddress

addresses = [
    StructuralAddress(0, 1, 10, 0),
    StructuralAddress(1, 1, 10, 1),
    StructuralAddress(2, 2, 20, 0),
    StructuralAddress(3, 2, 20, 1),
]
arr = torch.tensor([a.as_vector() for a in addresses], dtype=torch.float32)
D = arr[:, None, :] - arr[None, :, :]
mask = D[..., 1].abs() + (D[..., 2] != 0).float() + D[..., 3].abs()
plt.imshow(mask.numpy(), cmap='viridis')
plt.colorbar(label='structural distance')
plt.xticks(range(len(addresses)), [str(i) for i in range(len(addresses))])
plt.yticks(range(len(addresses)), [str(i) for i in range(len(addresses))])
plt.title('SGAM structural bias matrix')
plt.show()

In [ ]:
# Section 12: Full HiSEModel
import torch
from hise.config import HiSAConfig
from hise.model.hise_model import HiSEEmbeddingModel

model = HiSEEmbeddingModel(embedding_dim=768, config=HiSAConfig(d_model=768))
embeddings = torch.randn(4, 768)
structural_address = {
    'm': torch.tensor([0, 1, 2, 3], dtype=torch.long),
    'd': torch.tensor([1, 1, 2, 2], dtype=torch.long),
    'p': [[[]], [[]], [[]], [[]]],
    's': torch.tensor([0, 1, 0, 2], dtype=torch.long),
}
out = model(embeddings, structural_address)
print('HiSE output shape =', tuple(out.shape))
print('HiSE output mean =', out.mean().item())

In [ ]:
# Section 13: Run all 3 critical tests together
import torch
from hise.config import HiSAConfig
from hise.encoding.path_encoder import PathEncoder
from hise.encoding.subspace import DirectSumSubspace
from hise.encoding.utils import apply_rope, rope_freqs

results = {}
config = HiSAConfig(d_model=768)
model = DirectSumSubspace(config)
x = torch.randn(2, 16, 768)
positions = torch.arange(16).unsqueeze(0).expand(2, -1)
addr_zero = {'m': positions, 'd': torch.zeros(2, 16, dtype=torch.long), 'p': [[[] for _ in range(16)] for _ in range(2)], 's': torch.zeros(2, 16, dtype=torch.long)}
rope_out = apply_rope(x[..., :config.d_m], rope_freqs(config.d_m, config.rope_base, d_model=config.d_model), positions)
hisa_out = model(x, addr_zero)
results['degenerate'] = bool(torch.allclose(hisa_out[..., :config.d_m], rope_out, atol=1e-6))
encoder = PathEncoder(d_struct=154)
rot_a = encoder.encode_path([0, 1, 0])
rot_b = encoder.encode_path([0, 0, 1])
results['collision'] = bool(not torch.allclose(rot_a, rot_b, atol=1e-6))
parent_path = [0, 1]
child_sibling_idx = 2
child_path = parent_path + [child_sibling_idx]
parent_rot = encoder.encode_path(parent_path)
child_rot = encoder.encode_path(child_path)
recovered = encoder.get_parent_rotation(child_rot, child_sibling_idx)
results['invertible'] = bool(torch.allclose(parent_rot, recovered, atol=1e-5))
print(results)

In [ ]:
# Section 14: InfoNCE loss (tau=0.07)
import torch

def info_nce(z, z_pos, temperature=0.07):
    z = z / z.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    z_pos = z_pos / z_pos.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    logits = z @ z_pos.T / temperature
    return torch.nn.functional.cross_entropy(logits, torch.arange(z.shape[0]))

anchor = torch.randn(8, 768)
positive = anchor + 0.1 * torch.randn_like(anchor)
loss = info_nce(anchor, positive, temperature=0.07)
print('InfoNCE loss =', float(loss))

## Section 15: Next steps checklist

- [ ] Build `02_parsers.ipynb` for JSON/HTML/table parsers
- [ ] Build `03_data_pipeline.ipynb` for triplet generation and hard-negative curriculum
- [ ] Build `04_training.ipynb` for frozen-encoder fine-tuning with InfoNCE
- [ ] Add full benchmark against vanilla BGE / Nomic on structure-sensitive retrieval
- [ ] Compare mrr@k, nDCG@10, SDS, and query latency
- [ ] Convert the best notebook pipeline into a reusable training script

This notebook is the pre-attention core. Once it passes, everything above it (SGAM, parser integration, and training loop) is built on a stable foundation.